<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_04_4_batch_norm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>


# T81-558: Anwendungen tiefer neuronaler Netzwerke

**Modul 4: Training für tabellarische Daten**

- Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
– Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).


# Modul 4 Material

- Teil 4.1: Verwenden der K-fachen Kreuzvalidierung mit PyTorch [[Video]](https://www.youtube.com/watch?v=Q8ZQNvZwsNE&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=Q8ZQNvZwsNE&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.2: Trainingspläne für PyTorch [[Video]](https://www.youtube.com/watch?v=lMMlbmfvKDQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=lMMlbmfvKDQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.3: Dropout-Regularisierung [[Video]](https://www.youtube.com/watch?v=4ixjgw6Q42U&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=4ixjgw6Q42U&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- **Teil 4.4: Batch-Normalisierung** [[Video]](https://www.youtube.com/watch?v=1U5nOKh9OLQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=1U5nOKh9OLQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.5: RAPIDS für tabellarische Daten [[Video]](https://www.youtube.com/watch?v=KgoXuhG_kfs&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=KgoXuhG_kfs&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)


# Google CoLab-Anweisungen

Der folgende Code stellt sicher, dass Google CoLab ausgeführt wird und ordnet bei Bedarf Google Drive zu. Wir initialisieren das PyTorch-Gerät auch entweder auf GPU/MPS (falls verfügbar) oder CPU.


In [1]:
import copy
import torch

try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Nutzen Sie eine GPU oder MPS (Apple), sofern verfügbar. (siehe Modul 3.2)
import torch

has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwendetes Gerät: {device}")


# Frühzeitiges Stoppen (siehe Modul 3.4)
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_model = None
        self.best_loss = None
        self.counter = 0
        self.status = ""

    def __call__(self, model, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
        elif self.best_loss - val_loss >= self.min_delta:
            self.best_model = copy.deepcopy(model.state_dict())
            self.best_loss = val_loss
            self.counter = 0
            self.status = f"Improvement found, counter reset to {self.counter}"
        else:
            self.counter += 1
            self.status = f"No improvement in the last {self.counter} epochs"
            if self.counter >= self.patience:
                self.status = f"Early stopping triggered after {self.counter} epochs."
                if self.restore_best_weights:
                    model.load_state_dict(self.best_model)
                return True
        return False

Note: not using Google CoLab
Using device: mps


# Teil 4.4: Batch-Normalisierung

In diesem Abschnitt werden wir uns eingehend mit einem fortgeschrittenen Konzept zum Trainieren neuronaler Netzwerke befassen, das als Batch-Normalisierung bekannt ist. Das Ziel der Batch-Normalisierung besteht darin, das Lernen zu beschleunigen und den Lernprozess zu stabilisieren. Derselbe Datensatz, der zuvor in diesem Kapitel verwendet wurde. Um die Batch-Normalisierung zu implementieren, werden wir sowohl die Vorverarbeitung als auch die Einrichtung eines neuronalen Netzwerks ändern. Die ursprüngliche Vorverarbeitungsmethode verwendete eine Z-Score-Standardisierung, um die Daten für den Trainingsprozess besser geeignet zu machen. Das Modell selbst war ein einfaches Feedforward-Neuronales Netzwerk, das mit der nn.Sequential-API von PyTorch erstellt wurde.

Im überarbeiteten Code haben wir zwei wesentliche Änderungen vorgenommen. Die erste Änderung betraf die Vorverarbeitung der Daten. Da die Batch-Normalisierung die Auswirkungen von Änderungen der Eingabeverteilung (interne Kovariatenverschiebung) verringern kann, können wir den Schritt der Z-Score-Normalisierung entfernen. Die Batch-Normalisierung macht das Netzwerk tendenziell weniger empfindlich gegenüber dem Umfang und der Verteilung seiner Eingaben und minimiert so den Bedarf an manueller, sorgfältiger Datennormalisierung.

Die zweite und wichtigste Änderung wurde in der Architektur des neuronalen Netzwerks selbst vorgenommen. Wir haben mithilfe der Funktion nn.BatchNorm1d() Batch-Normalisierungsebenen in unser Modell eingefügt. Es ist wichtig zu beachten, dass die Batch-Normalisierungsebenen normalerweise nach den linearen (oder Faltungs- für ConvNets) Ebenen, aber vor der Aktivierungsfunktion hinzugefügt werden. In unserem Fall ist die Reihenfolge: Linear -> BatchNorm -> ReLU.

Die Batch-Normalisierungsebenen normalisieren die Aktivierungen und Gradienten, die sich durch ein neuronales Netzwerk ausbreiten, wodurch das Modelltraining effizienter wird. Dies kann sogar einen leichten Regularisierungseffekt haben, ähnlich wie Dropout.

Bedenken Sie, dass die Verwendung der Batch-Normalisierung aufgrund der zusätzlichen Komplexität des Modells möglicherweise zusätzliche Rechenressourcen erfordert, aber häufig zu einer erheblichen Leistungssteigerung führt, die die zusätzliche Rechenzeit mehr als ausgleicht.

Zusammenfassend lässt sich sagen, dass die Einführung der Batch-Normalisierung in unserem neuronalen Netzwerkmodell die Vorverarbeitung vereinfacht und möglicherweise die Trainingsgeschwindigkeit, Stabilität und Gesamtleistung des Modells verbessern kann. Sehen wir uns nun den Code genauer an und schauen wir, wie diese Verbesserungen in der Praxis umgesetzt werden.

Es folgt der geänderte Code.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold

# Lesen des Datensatzes
df = read_csv(
    "https://data.heatonresearch.com/data/t81-558/jh-simple-dataset.csv",
    na_values=["NA", "?"],
)

# Dummies für den Job generieren
df = concat([df, get_dummies(df["job"], prefix="job", dtype=int)], axis=1)
df.drop("job", axis=1, inplace=True)

# Dummies für den Bereich generieren
df = concat([df, get_dummies(df["area"], prefix="area", dtype=int)], axis=1)
df.drop("area", axis=1, inplace=True)

# Dummies für Produkte generieren
df = concat([df, get_dummies(df["product"], prefix="product", dtype=int)], axis=1)
df.drop("product", axis=1, inplace=True)

# Fehlende Werte für Einkommen
med = df["income"].median()
df["income"] = df["income"].fillna(med)

# In PyTorch-Tensoren konvertieren
x_columns = df.columns.drop(["age", "id"])
x = torch.tensor(df[x_columns].values, dtype=torch.float32, device=device)
y = torch.tensor(df["age"].values, dtype=torch.float32, device=device).view(-1, 1)

# Legen Sie einen Zufallsstartwert für die Reproduzierbarkeit fest
torch.manual_seed(42)

# Kreuzvalidierung
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Parameter für vorzeitiges Stoppen
patience = 10

fold = 0
for train_idx, test_idx in kf.split(x):
    fold += 1
    print(f"Fold # {falten}")

    x_train, x_test = x[train_idx], x[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # PyTorch DataLoader
    train_dataset = TensorDataset(x_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    # Erstellen des Modells und des Optimierers
    model = nn.Sequential(
        nn.Linear(x.shape[1], 20),
        nn.BatchNorm1d(20),  # BatchNorm-Ebene
        nn.ReLU(),
        nn.Linear(20, 10),
        nn.BatchNorm1d(10),  # BatchNorm-Ebene
        nn.ReLU(),
        nn.Linear(10, 1),
    )
    model = torch.compile(model, backend="aot_eager").to(device)

    optimizer = optim.Adam(model.parameters())
    loss_fn = nn.MSELoss()

    # Variablen für frühzeitiges Stoppen
    best_loss = float("inf")
    early_stopping_counter = 0

    # Trainingsschleife
    EPOCHS = 500
    epoch = 0
    done = False
    es = EarlyStopping()

    while not done and epoch < EPOCHS:
        epoch += 1
        model.train()
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(x_batch)
            loss = loss_fn(output, y_batch)
            loss.backward()
            optimizer.step()

        # Validierung
        model.eval()
        with torch.no_grad():
            val_output = model(x_test)
            val_loss = loss_fn(val_output, y_test)

        if es(model, val_loss):
            done = True

    print(
        f"Epoch {epoch}/{EPOCHS}, Validation Loss: " f"{val_loss.item()}, {es.status}"
    )

# Abschließende Bewertung
model.eval()
with torch.no_grad():
    oos_pred = model(x_test)
score = torch.sqrt(loss_fn(oos_pred, y_test)).item()
print(f"Faltungsbewertung (RMSE): {score}")

Fold #1


/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/autograd/__init__.py:394: UserWarning: Error detected in LinearBackward0. Traceback of forward call that caused the error:
  File "/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/nn/modules/container.py", line 215, in forward
    input = module(input)
 (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1702400227158/work/torch/csrc/autograd/python_anomaly_mode.cpp:119.)
  result = Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[2024-01-07 07:29:39,418] [0/2] torch._dynamo.exc: [WARNING] Backend compiler failed with a fake tensor exception at 
[2024-01-07 07:29:39,418] [0/2] torch._dynamo.exc: [WARNING]   File "/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/nn/modules/container.py", line 216, in forward
[2024-01-07 07:29:39,418] [0/2] torch._dynamo.exc: [WARNING]     return input
[2024-01-07 07:29:39,418] [

Epoch 24/500, Validation Loss: 347.8447265625, Early stopping triggered after 5 epochs.
Fold #2
Epoch 25/500, Validation Loss: 1626.8416748046875, Early stopping triggered after 5 epochs.
Fold #3
Epoch 15/500, Validation Loss: 891.66796875, Early stopping triggered after 5 epochs.
Fold #4
Epoch 39/500, Validation Loss: 100.74588012695312, Early stopping triggered after 5 epochs.
Fold #5
Epoch 15/500, Validation Loss: 999.7000732421875, Early stopping triggered after 5 epochs.
Fold score (RMSE): 24.599931716918945


Batch-Normalisierung kann die Leistung eines Modells oft verbessern, ist aber keine Garantie dafür. Sie kann von vielen Faktoren abhängen, wie dem spezifischen Problem, das behandelt wird, den Daten, der Architektur des Modells und dem spezifischen Trainingsregime.

* Datensatz: Die Batch-Normalisierung ist besonders nützlich, wenn Sie mit hochdimensionalen Daten arbeiten, und ist bei größeren Datensätzen tendenziell effektiver. Wenn Ihr Datensatz klein oder einfach ist, macht die Batch-Normalisierung möglicherweise keinen großen Unterschied und kann sogar zu einer leichten Leistungsminderung führen.

* Modellkomplexität: Bei einfacheren Modellen führt die Batch-Normalisierung möglicherweise nicht zu einer signifikanten Leistungssteigerung und kann die Leistung sogar leicht verschlechtern, da sie zusätzliche Komplexität und Berechnungen mit sich bringt.

* Trainingsparameter: Die Verbesserung durch Batch-Normalisierung hängt auch von den anderen von Ihnen verwendeten Hyperparametern ab, wie etwa der Lernrate und der Batch-Größe. Wenn das ursprüngliche Modell mit diesen Parametern bereits gut optimiert war, hilft die Hinzufügung der Batch-Normalisierung möglicherweise nicht viel und kann möglicherweise sogar die Leistung beeinträchtigen.

Mit anderen Worten: Obwohl die Batch-Normalisierung im Allgemeinen eine nützliche Technik zur Verbesserung der Modellleistung und -stabilität ist, ist sie kein Allheilmittel und führt nicht immer zu einer Verbesserung. Es ist immer eine gute Idee, mit verschiedenen Architekturen und Techniken zu experimentieren, um herauszufinden, was für Ihr spezifisches Problem am besten funktioniert.